# Compare GPT and Jev trying to guess how much things cost

## Give them 200 Amazon items and ask them to guess the cost

Using price buckets for Jev

In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np

In [ ]:
load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
train, val, test = Item.from_hub("ed-donner/items_full")
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# First the frontier

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
print(test[0].summary)

In [ ]:
def nano(item):
    response = completion(model="openrouter/openai/gpt-4.1-nano", messages=messages_for(item), seed=42)
    return response.choices[0].message.content, response._hidden_params["response_cost"]



In [ ]:
nano(test[0])[0]

In [ ]:
test[0].price

In [ ]:
evaluate(nano, test)

In [ ]:
def luna(item):
    response = completion(model="openrouter/openai/gpt-5.6-luna", messages=messages_for(item), seed=42, reasoning_effort="none")
    return response.choices[0].message.content, response._hidden_params["response_cost"]
evaluate(luna, test)

# And now - Jev

Jev is a System One model from TypeSafe: it doesn't generate text, it returns a probability distribution over options that we define up front.

There's no continuous number to ask for, so we split the price range into $20 wide buckets and ask one Choice question. Jev picks a bucket and hands back a probability for every one of them; we do the arithmetic ourselves in code.

It's served through OpenRouter's Decisions API, which is a different endpoint to chat completions, so litellm isn't involved here.

In [ ]:
from openrouter import OpenRouter

BUCKET_WIDTH = 20
TOP_PRICE = 1000

buckets = [(low, low + BUCKET_WIDTH) for low in range(0, TOP_PRICE, BUCKET_WIDTH)]
criteria = {f"${low}-${high}": f"The product sells for between ${low} and ${high}" for low, high in buckets}
midpoints = {f"${low}-${high}": (low + high) / 2 for low, high in buckets}

# The Decisions API lives outside the usual /api/v1 prefix, hence the server_url
openrouter = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"], server_url="https://openrouter.ai")

print(f"{len(criteria)} buckets, from {buckets[0]} to {buckets[-1]}")

In [ ]:
questions={
            "price": {
                "type": "choice",
                "instructions": "How much does this product cost at retail in US dollars?",
                "criteria": criteria,
            }
        }

In [ ]:
def jev(item):
    response = openrouter.alpha.decisions.create(model="typesafe/jev-1.13", state=item.summary, questions=questions)
    answer = response.answers["price"]
    return midpoints[answer.choice], response.usage.cost

In [ ]:
jev(test[0])[0]

In [ ]:
test[0].price

In [ ]:
evaluate(jev, test)

## Reading the distribution better

Two changes, neither of which tells Jev anything about how prices are distributed:

**Ask at two resolutions.** The same question with $20 buckets and again with $100 buckets, both in one request so they cost one call. Spread each answer over the $20 lattice, multiply, renormalise. Where the two resolutions disagree the mass cancels out.

**Read the median, not the argmax or the mean.** On a distribution this spread out, the top bucket can win an argmax with 15% of the mass; it can't be the median unless real mass sits up there. The mean has the opposite problem - a long tail of buckets at the rounding floor of 0.01 drags it upwards.

In [ ]:
COARSE_WIDTH = 100

coarse_buckets = [(low, low + COARSE_WIDTH) for low in range(0, TOP_PRICE, COARSE_WIDTH)]
coarse_criteria = {f"${low}-${high}": f"The product sells for between ${low} and ${high}" for low, high in coarse_buckets}

bucket_names = list(criteria)
coarse_names = list(coarse_criteria)
lows = np.array([low for low, high in buckets], dtype=float)

PRICE_QUESTION = "How much does this product cost at retail in US dollars?"


def spread(probabilities, names, width):
    """Spread one grid's probabilities evenly across the $20 buckets each of them covers."""
    per_bucket = width // BUCKET_WIDTH
    values = np.array([probabilities.get(name, 0.0) for name in names])
    return np.repeat(values / per_bucket, per_bucket)


def median_price(distribution):
    """The price at which the distribution reaches half its mass, interpolated inside the bucket."""
    distribution = distribution / distribution.sum()
    cumulative = np.cumsum(distribution)
    index = min(int(np.searchsorted(cumulative, 0.5)), len(distribution) - 1)
    below = cumulative[index - 1] if index else 0.0
    if distribution[index] == 0:
        return float(lows[index] + BUCKET_WIDTH / 2)
    return float(lows[index] + (0.5 - below) / distribution[index] * BUCKET_WIDTH)

In [ ]:
def jev_two_pass_median(item):
    """Both resolutions in a single request; combine the two distributions and take the median."""
    response = openrouter.alpha.decisions.create(
        model="typesafe/jev-1.13",
        state=item.summary,
        questions={
            "fine": {"type": "choice", "instructions": PRICE_QUESTION, "criteria": criteria},
            "coarse": {"type": "choice", "instructions": PRICE_QUESTION, "criteria": coarse_criteria},
        },
    )
    answers = response.answers

    fine = spread(answers["fine"].probabilities, bucket_names, BUCKET_WIDTH)
    coarse = spread(answers["coarse"].probabilities, coarse_names, COARSE_WIDTH)

    combined = fine * coarse
    if combined.sum() == 0:
        combined = fine  # the two grids agreed on nothing, so fall back to the fine one alone

    return median_price(combined), response.usage.cost

In [ ]:
estimate, cost = jev_two_pass_median(test[0])
print(f"{estimate:.2f} vs actual {test[0].price:.2f}, cost ${cost:.6f}")

In [ ]:
evaluate(jev_two_pass_median, test)